In [1]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('../data/db/ecommerce.db')

In [2]:
# 01 GROUP BY - basic aggregation:

#Country-wise total transactions count
q1 = pd.read_sql(""" 
    SELECT Country, COUNT(*) as total_transactions
    FROM transactions
    GROUP BY Country
    ORDER BY total_transactions DESC
""", conn) #COUNT(*) ek aggregate function hai jo har country ki total rows(transactions) ki counting karta hai.
#as total_transactions ye us naye counting wale column ko ek clean total_transactions name deta hai.
#GROUP BY ye sari transactions ko countries ke according alag-alag gruops me divide kar deta hai taki har country ka alag-alag count nikal sake.
print(q1)

                 Country  total_transactions
0         United Kingdom              981330
1                   EIRE               17866
2                Germany               17624
3                 France               14330
4            Netherlands                5140
5                  Spain                3811
6            Switzerland                3189
7                Belgium                3123
8               Portugal                2620
9              Australia                1913
10       Channel Islands                1664
11                 Italy                1534
12                Norway                1455
13                Sweden                1364
14                Cyprus                1176
15               Finland                1049
16               Austria                 938
17               Denmark                 817
18           Unspecified                 756
19                Greece                 663
20                 Japan                 582
21        

In [3]:
# Customer-wise total spend (Qunatity*Price), is query se pata chelga ki kis customer ne totlal kitna paisa spend kiya hai.
q2 = pd.read_sql("""
    SELECT "Customer ID", SUM(Quantity * Price) as total_spend
    FROM transactions
    WHERE "Customer ID" IS NOT NULL AND Quantity > 0
    GROUP BY "Customer ID"
    ORDER BY total_spend DESC
    LIMIT 10
""", conn)
print(q2)

   Customer ID  total_spend
0      18102.0    608821.65
1      14646.0    528602.52
2      14156.0    313946.37
3      14911.0    295972.63
4      17450.0    246973.09
5      13694.0    196482.81
6      17511.0    175603.55
7      16446.0    168472.50
8      16684.0    147142.77
9      12415.0    144458.37


In [4]:
#02 Multiple aggregate functions ek saath

In [5]:
q3 = pd.read_sql("""
    SELECT Country,
           COUNT(DISTINCT Invoice) as total_orders,
           COUNT(DISTINCT "Customer ID") as unique_customers,
           SUM(Quantity) as avg_price
        FROM transactions
        WHERE Quantity > 0
        GROUP BY Country
        ORDER BY total_orders DESC
""", conn)

print(q3)

                 Country  total_orders  unique_customers  avg_price
0         United Kingdom         38401              5353    9634240
1                Germany           789               107     228003
2                   EIRE           626                 5     340564
3                 France           622                95     275288
4            Netherlands           229                22     384617
5                  Spain           154                41      50807
6                Belgium           149                29      35312
7                 Sweden           105                19      88650
8               Portugal            95                24      28409
9              Australia            95                15     104398
10           Switzerland            93                22      52885
11                 Italy            65                17      15501
12               Finland            57                14      14375
13       Channel Islands            55          

In [6]:
#03 HAVING- GROUP BY ke baad filter lagana

#sirf wo Countries jinke 100 se jyada unique customers hai
q4 = pd.read_sql("""
    SELECT Country, COUNT(DISTINCT "Customer ID") as unique_customers
    FROM transactions
    GROUP BY Country
    HAVING unique_customers > 100
    ORDER BY unique_customers DESC
""", conn)
print(q4)

          Country  unique_customers
0  United Kingdom              5410
1         Germany               107


In [7]:

#sirf wo customers jinka total spend 5000 se jyada hai (high-value customers - churn analysis me kaam karega)
q5 = pd.read_sql("""
    SELECT "Customer ID", SUM(Quantity * Price) as total_spend
    FROM transactions
    WHERE "Customer ID" IS NOT NULL AND Quantity > 0
    GROUP BY "Customer ID"
    HAVING total_spend > 5000
    ORDER BY total_spend DESC
""", conn)
print(q5.shape)
print(q5.head())

(670, 2)
   Customer ID  total_spend
0      18102.0    608821.65
1      14646.0    528602.52
2      14156.0    313946.37
3      14911.0    295972.63
4      17450.0    246973.09


In [8]:
#04-Product-level analysis

In [9]:
#top 10 sabse jyada bikne wale products
q6 = pd.read_sql("""
    SELECT StockCode, Description, SUM(Quantity) as total_sold
    FROM transactions
    WHERE Quantity > 0
    GROUP BY StockCode, Description
    ORDER BY total_sold DESC
    LIMIT 10
""", conn)
print(q6)

  StockCode                         Description  total_sold
0     84077   WORLD WAR 2 GLIDERS ASSTD DESIGNS      110249
1    85123A  WHITE HANGING HEART T-LIGHT HOLDER       96091
2     84879       ASSORTED COLOUR BIRD ORNAMENT       81817
3     23843         PAPER CRAFT , LITTLE BIRDIE       80995
4    85099B             JUMBO BAG RED RETROSPOT       78866
5     23166      MEDIUM CERAMIC TOP STORAGE JAR       78033
6     17003                 BROCADE RING PURSE        71440
7     21977  PACK OF 60 PINK PAISLEY CAKE CASES       56794
8     84991         60 TEATIME FAIRY CAKE CASES       54716
9     22197                SMALL POPCORN HOLDER       49984


In [10]:
conn.close()